In [9]:
!pip install selenium webdriver-manager pandas tqdm lxml cloudscraper playwright

     |████████████████████████████████| 38.6 MB 59.5 MB/s eta 0:00:01
     |████████████████████████████████| 267 kB 17.6 MB/s eta 0:00:01
You should consider upgrading via the '/Users/tonyliu/Documents/DriftNet/data_centers/driftnet/bin/python3 -m pip install --upgrade pip' command.


In [21]:
!pip install undetected-chromedriver selenium webdriver-manager

     |████████████████████████████████| 65 kB 4.4 MB/s eta 0:00:01
     |████████████████████████████████| 173 kB 9.1 MB/s eta 0:00:01
Using legacy 'setup.py install' for undetected-chromedriver, since package 'wheel' is not installed.
    Running setup.py install for undetected-chromedriver ... done
You should consider upgrading via the '/Users/tonyliu/Documents/DriftNet/data_centers/driftnet/bin/python3 -m pip install --upgrade pip' command.


In [92]:
# scrape_usa_datacenters_with_restart.py
import json, random, time, tempfile, shutil, pandas as pd
import undetected_chromedriver as uc
from selenium.webdriver.common.by        import By
from selenium.webdriver.chrome.options   import Options
from selenium.webdriver.support.ui       import WebDriverWait
from selenium.webdriver.support          import expected_conditions as EC
from selenium.common.exceptions          import (
    TimeoutException, NoSuchWindowException, WebDriverException
)

COUNTRY_URL = "https://www.datacentermap.com/usa/"
OUTFILE     = "usa_datacenters.csv"

TIMEOUT_SEC = 2         # keep it realistic – 2 s is rarely enough
MAX_RETRY   = 20

UA_POOL = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/137.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_4) AppleWebKit/605.1.15 "
    "(KHTML, like Gecko) Version/17.4 Safari/605.1.15",
]

# ───────────────────────── helpers ──────────────────────────
def spin_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1280,900")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--no-sandbox")
    # opts.add_argument(f"--user-agent={random.choice(UA_POOL)}")  # optional
    profile_dir = tempfile.mkdtemp(prefix="dcm-usa-")
    opts.add_argument(f"--user-data-dir={profile_dir}")
    opts.add_argument("--blink-settings=imagesEnabled=false")
    opts.add_argument("--disable-dev-shm-usage")   
    return uc.Chrome(options=opts, version_main=137), profile_dir

def restart_driver(old_drv, old_profile):
    try:
        old_drv.quit()
    finally:
        shutil.rmtree(old_profile, ignore_errors=True)
    return spin_driver()

def wait_for_next_data(drv):
    WebDriverWait(drv, TIMEOUT_SEC).until(
        lambda d: '"mapdata"' in d.find_element(By.ID, "__NEXT_DATA__")
                                         .get_attribute("textContent")
    )

def get_with_retry(drv, profile_dir, url):
    for attempt in range(1, MAX_RETRY + 1):
        try:
            drv.get(url)
            wait_for_next_data(drv)
            return drv, profile_dir
        except (TimeoutException, NoSuchWindowException, WebDriverException) as e:
            print(f"   ⚠ blocker / crash ({e.__class__.__name__}) – "
                  f"restart #{attempt}/{MAX_RETRY}")
            drv, profile_dir = restart_driver(drv, profile_dir)
    raise RuntimeError(f"Giving up after {MAX_RETRY} failed attempts → {url}")

def human_pause(a=0.7, b=1.8):
    time.sleep(random.uniform(a, b))

# ─────────────────────────── scrape ────────────────────────────
drv, profile = spin_driver()
all_rows = []

try:
    # ①  COUNTRY → list of states
    drv, profile = get_with_retry(drv, profile, COUNTRY_URL)
    country_json = json.loads(
        drv.find_element(By.ID, "__NEXT_DATA__").get_attribute("textContent")
    )
    states = [
        f"https://www.datacentermap.com/usa/{s['properties']['link']}/"
        for s in country_json["props"]["pageProps"]["mapdata"]["geos"]
        if s["properties"]["datacenters"] > 0
    ]
    print(f"▶ Found {len(states)} US states with data-centres")

    # ②  STATE loop
    for s_i, state_url in enumerate(states, 1):
        state_slug = state_url.rstrip("/").split("/")[-1]
        print(f"\n════ State {s_i}/{len(states)}  {state_slug}")

        drv, profile = get_with_retry(drv, profile, state_url)
        state_json = json.loads(
            drv.find_element(By.ID, "__NEXT_DATA__").get_attribute("textContent")
        )

        markets = [
            f"https://www.datacentermap.com/usa/{state_slug}/{m['properties']['link']}/"
            for m in state_json["props"]["pageProps"]["mapdata"]["geos"]
            if m["properties"]["datacenters"] > 0
        ]
        print(f"   ↳ {len(markets)} markets")

        # ③  MARKET loop (same logic you had for California)
        for m_i, market_url in enumerate(markets, 1):
            drv, profile = get_with_retry(drv, profile, market_url)
            city_json = json.loads(
                drv.find_element(By.ID, "__NEXT_DATA__").get_attribute("textContent")
            )
            dcs = city_json["props"]["pageProps"]["mapdata"]["dcs"]

            for feat in dcs:              # ← unchanged inner try/except
                try:
                    p = feat["properties"]
                    all_rows.append({
                        "id":          p["id"],
                        "name":        p["name"],
                        "company":     p.get("companyname"),
                        "type":        p["type"],
                        "address":     p["address"],
                        "city":        p["city"],
                        "postal":      p["postal"],
                        "state":       p["state"],
                        "lat":         feat["geometry"]["coordinates"][1],
                        "lon":         feat["geometry"]["coordinates"][0],
                        "profile_url": "https://www.datacentermap.com" + p["url"],
                    })
                except Exception as e:
                    print(f"      ⚠ Error processing → {e}")

            print(f"      • {m_i:>2}/{len(markets)} "
                  f"{market_url.split('/')[-2]:<18} → {len(dcs):>3} facilities")
            human_pause()

finally:
    drv.quit()
    shutil.rmtree(profile, ignore_errors=True)

# ④  SAVE
pd.DataFrame(all_rows).drop_duplicates("id").to_csv(OUTFILE, index=False)
print(f"\n✅  saved {len(all_rows)} total rows → {OUTFILE}")


   ⚠ blocker / crash (NoSuchWindowException) – restart #1/20
▶ Found 51 US states with data-centres

════ State 1/51  california
   ↳ 29 markets
      •  1/29 los-angeles        →  70 facilities
      ⚠ Error processing → 'id'
      •  2/29 san-jose           →  45 facilities
      •  3/29 fremont            →   6 facilities
      •  4/29 san-francisco      →  15 facilities
      •  5/29 santa-clara        →  75 facilities
      •  6/29 palo-alto          →   3 facilities
      •  7/29 sacramento         →  22 facilities
      •  8/29 oakland            →   1 facilities
      •  9/29 san-diego          →  14 facilities
      • 10/29 irvine             →  15 facilities
      • 11/29 fresno             →   3 facilities
      • 12/29 tustin             →   1 facilities
      • 13/29 mountain-view      →   1 facilities
      • 14/29 san-luis-obispo    →   4 facilities
      • 15/29 modesto            →   2 facilities
      • 16/29 el-segundo         →   3 facilities
      • 17/29 riverside

In [4]:
# scrape_usa_datacenters_with_profiles.py
import json, random, time, tempfile, shutil, pandas as pd
import undetected_chromedriver as uc
import urllib3                                           # ← new
from urllib3.exceptions import MaxRetryError             # ← new
from http.client import RemoteDisconnected  
from selenium.webdriver.common.by        import By
from selenium.webdriver.chrome.options   import Options
from selenium.webdriver.support.ui       import WebDriverWait
from selenium.webdriver.support          import expected_conditions as EC
from selenium.common.exceptions          import (
    TimeoutException, NoSuchWindowException, WebDriverException
)

COUNTRY_URL = "https://www.datacentermap.com/usa/"
OUTFILE     = "usa_datacenters_profiles.csv"

TIMEOUT_SEC = 3
MAX_RETRY   = 20

UA_POOL = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/137.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_4) AppleWebKit/605.1.15 "
    "(KHTML, like Gecko) Version/17.4 Safari/605.1.15",
]

## ───────────────────────── helpers ──────────────────────────
RESTART_ERRORS = (
    TimeoutException,
    NoSuchWindowException,
    WebDriverException,
    MaxRetryError,
    RemoteDisconnected,
    ConnectionRefusedError,
)

def human_pause(a=0.7, b=1.8):
    time.sleep(random.uniform(a, b))

def is_blocked(drv) -> bool:
    """Heuristics for Cloudflare / 429 pages."""
    title = drv.title.lower()
    body  = drv.page_source[:2000].lower()     # first couple kB are enough
    return (
        "429"          in title
        or "too many requests" in title
        or "cloudflare" in body and "verify" in body
    )

def spin_driver():
    opts = Options()
    opts.add_argument("--headless=new")   # comment out to watch
    opts.add_argument("--window-size=1280,900")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--blink-settings=imagesEnabled=false")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument(f"--user-agent={random.choice(UA_POOL)}")

    profile_dir = tempfile.mkdtemp(prefix="dcm-usa-")
    opts.add_argument(f"--user-data-dir={profile_dir}")

    # ── THIS LINE changed:  use_subprocess=True
    drv = uc.Chrome(
        options=opts, version_main=137, use_subprocess=True
    )
    return drv, profile_dir

def restart_driver(old_drv, old_profile):
    try:
        old_drv.quit()
    finally:
        shutil.rmtree(old_profile, ignore_errors=True)
    return spin_driver()

def wait_for_next_data(drv):
    WebDriverWait(drv, TIMEOUT_SEC).until(
        lambda d: d.find_element(By.ID, "__NEXT_DATA__")
    )

def get_with_retry(drv, profile_dir, url, wait_json=True):
    """
    Open `url`, retrying & restarting the browser whenever we hit:
      • selenium crashes (Timeout / NoSuchWindow / …)
      • dev-tools socket errors (MaxRetryError, RemoteDisconnected, …)
      • Cloudflare / 429 free-page quota
    """
    for attempt in range(1, MAX_RETRY + 1):
        try:
            drv.get(url)

            # ── 429 quota page? ───────────────────────────────
            if is_blocked(drv):
                raise ValueError("429 page")

            if wait_json:
                wait_for_next_data(drv)
            return drv, profile_dir                      # success ✅

        except (*RESTART_ERRORS, ValueError) as e:
            lbl = "429" if isinstance(e, ValueError) else e.__class__.__name__
            print(f"   ↺ restart ({lbl})  {attempt}/{MAX_RETRY}")
            drv, profile_dir = restart_driver(drv, profile_dir)

    # exhausted retries
    raise RuntimeError(f"Giving up after {MAX_RETRY} attempts → {url}")

# ----------  profile-page helper  --------------------------------
def _flatten_specs(dc):
    flat = {}
    for block in (
        "meta_power", "meta_building", "meta_capacity",
        "meta_security", "meta_amenities", "meta_standards",
    ):
        for k, v in dc.get(block, {}).items():
            flat[f"{block}_{k}"] = v
    return flat

def extract_profile(drv, session_dir, url):
    """
    Fetch *one* data-centre profile and return its extra fields.
    The same get_with_retry() logic guarantees we either get
    a valid JSON payload or an empty dict after MAX_RETRY attempts.
    """
    try:
        drv, session_dir = get_with_retry(
            drv, session_dir, url, wait_json=True
        )
        js = json.loads(
            drv.find_element(By.ID, "__NEXT_DATA__").get_attribute("textContent")
        )
        dc = js["props"]["pageProps"]["dc"]
        details = {
            "profile_status": dc["status"],
            "profile_stage":  dc["stage"],
            "description":    dc.get("description", ""),
        }
        details.update(_flatten_specs(dc))
        return details, drv, session_dir
    except Exception:
        # after MAX_RETRY the caller still gets an empty dict (row preserved)
        return {}, drv, session_dir

# ─────────────────────────── scrape ────────────────────────────
drv, profile = spin_driver()
all_rows = []

try:
    # ①  COUNTRY → states
    drv, profile = get_with_retry(drv, profile, COUNTRY_URL)
    country_json = json.loads(
        drv.find_element(By.ID, "__NEXT_DATA__").get_attribute("textContent")
    )
    states = [
        f"https://www.datacentermap.com/usa/{s['properties']['link']}/"
        for s in country_json["props"]["pageProps"]["mapdata"]["geos"]
        if s["properties"]["datacenters"] > 0
    ]
    print(f"▶ Found {len(states)} US states with data-centres")

    # ②  STATE loop
    for s_i, state_url in enumerate(states, 1):
        state_slug = state_url.rstrip("/").split("/")[-1]
        print(f"\n════ {s_i:02}/{len(states)}  {state_slug}")

        drv, profile = get_with_retry(drv, profile, state_url)
        state_json = json.loads(
            drv.find_element(By.ID, "__NEXT_DATA__").get_attribute("textContent")
        )

        markets = [
            f"https://www.datacentermap.com/usa/{state_slug}/{m['properties']['link']}/"
            for m in state_json["props"]["pageProps"]["mapdata"]["geos"]
            if m["properties"]["datacenters"] > 0
        ]
        print(f"   ↳ {len(markets)} markets")

        # ③  MARKET loop
        for m_i, market_url in enumerate(markets, 1):
            
            for attempt in range(1, MAX_RETRY + 1):
                drv, profile = get_with_retry(drv, profile, market_url)
                try:
                    city_json = json.loads(
                        drv.find_element(By.ID, "__NEXT_DATA__").get_attribute("textContent")
                    )
                    dcs = city_json["props"]["pageProps"]["mapdata"]["dcs"]
                    break  # success ✅
                except Exception as e:
                    print(f"   ⚠ blocker / crash ({e.__class__.__name__}) – "
                          f"restart #{attempt}/{MAX_RETRY}")
                    drv, profile = restart_driver(drv, profile)

            for feat in dcs:
                try:
                    p = feat["properties"]
                    base = dict(
                        id=p["id"],
                        name=p["name"],
                        company=p.get("companyname"),
                        type=p["type"],
                        address=p["address"],
                        city=p["city"],
                        postal=p["postal"],
                        state=p["state"],
                        lat=feat["geometry"]["coordinates"][1],
                        lon=feat["geometry"]["coordinates"][0],
                        profile_url="https://www.datacentermap.com" + p["url"],
                    )

                    # --- deep-dive: one extra page load -----------------
                    extra, drv, profile = extract_profile(
                        drv, profile, base["profile_url"]
                    )
                    base.update(extra)
                    all_rows.append(base)

                except Exception as e:
                    print(f"      ⚠ parse: {e}")

                human_pause(0.15, 0.35)   # tiny back-off between profiles

            print(
                f"      • {m_i:>2}/{len(markets)} "
                f"{market_url.split('/')[-2]:<18} → {len(dcs):>3} profiles"
            )

finally:
    drv.quit()
    shutil.rmtree(profile, ignore_errors=True)

# ④  SAVE
df = pd.DataFrame(all_rows).drop_duplicates("id")
df.to_csv(OUTFILE, index=False)
print(f"\n✅  saved {len(df)} enriched rows → {OUTFILE}")


▶ Found 51 US states with data-centres

════ 01/51  california
   ↳ 29 markets
      •  1/29 los-angeles        →  70 profiles
      ⚠ parse: 'id'
      •  2/29 san-jose           →  45 profiles
      •  3/29 fremont            →   6 profiles
      •  4/29 san-francisco      →  15 profiles
      •  5/29 santa-clara        →  75 profiles
      •  6/29 palo-alto          →   3 profiles
      •  7/29 sacramento         →  22 profiles
      •  8/29 oakland            →   1 profiles
      •  9/29 san-diego          →  14 profiles
      • 10/29 irvine             →  15 profiles
      • 11/29 fresno             →   3 profiles
      • 12/29 tustin             →   1 profiles
      • 13/29 mountain-view      →   1 profiles
   ⚠ blocker / crash (KeyError) – restart #1/20
      • 14/29 san-luis-obispo    →   4 profiles
   ⚠ blocker / crash (KeyError) – restart #1/20
      • 15/29 modesto            →   2 profiles
   ⚠ blocker / crash (KeyError) – restart #1/20
      • 16/29 el-segundo         →   

RuntimeError: Giving up after 20 attempts → https://www.datacentermap.com/usa/utah/santaquin/

In [7]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   Enrich existing “basic” USA data-centre list with profile  ║
# ║   details – crash-safe & resumable                           ║
# ╚══════════════════════════════════════════════════════════════╝
import json, random, time, tempfile, shutil, csv, pathlib
import pandas as pd
import undetected_chromedriver as uc
from selenium.webdriver.common.by      import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui     import WebDriverWait
from selenium.webdriver.support        import expected_conditions as EC
from selenium.common.exceptions        import (
    TimeoutException, NoSuchWindowException, WebDriverException)

# ------------------------------------------------------------------
BASIC_CSV   = "usa_datacenters.csv"          # ← your existing file
OUT_JSONL   = "usa_datacenters_profiles.jsonl"
TIMEOUT_SEC = 3
MAX_RETRY   = 20

UA_POOL = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_4) "
    "AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Safari/605.1.15",
]

# ────────────────────────── driver helpers ─────────────────────────
def spin_driver():
    opts = Options()
    #opts.add_argument("--headless=new")      # comment to watch the browser
    opts.add_argument("--window-size=1280,900")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--blink-settings=imagesEnabled=false")
    opts.add_argument(f"--user-agent={random.choice(UA_POOL)}")

    profile_dir = tempfile.mkdtemp(prefix="dcm-enrich-")
    opts.add_argument(f"--user-data-dir={profile_dir}")
    drv = uc.Chrome(options=opts, version_main=137)
    return drv, profile_dir

def restart_driver(drv, prof):
    try:
        drv.quit()
    finally:
        shutil.rmtree(prof, ignore_errors=True)
    return spin_driver()

def wait_for_next_data(drv):
    WebDriverWait(drv, TIMEOUT_SEC).until(
        lambda d: d.find_element(By.ID, "__NEXT_DATA__"))

# ─────────────────── profile-page extraction ───────────────────────
def flatten_meta(dc):
    out = {}
    for blk in (
        "meta_power", "meta_building", "meta_capacity",
        "meta_security", "meta_amenities", "meta_standards"):
        for k, v in dc.get(blk, {}).items():
            out[f"{blk}_{k}"] = v
    return out

def grab_profile(drv, prof_dir, url):
    """Return dict with extra fields - or {} on repeated failure."""
    for _ in range(MAX_RETRY):
        try:
            drv.get(url)
            wait_for_next_data(drv)

            js = json.loads(
                drv.find_element(By.ID, "__NEXT_DATA__")
                   .get_attribute("textContent"))
            dc = js["props"]["pageProps"].get("dc")
            if not dc:                          # hit quota page?
                raise ValueError("no 'dc' block")

            extra = {
                "profile_status": dc["status"],
                "profile_stage":  dc["stage"],
                "description":    dc.get("description", ""),
            }
            extra.update(flatten_meta(dc))
            return extra, drv, prof_dir

        except (TimeoutException, ValueError, WebDriverException):
            drv, prof_dir = restart_driver(drv, prof_dir)
            continue
    return {}, drv, prof_dir

# ───────────────────────  resume bookkeeping  ──────────────────────
done = set()
if pathlib.Path(OUT_JSONL).is_file():
    with open(OUT_JSONL, encoding="utf-8") as fh:
        for ln in fh:
            try:
                done.add(json.loads(ln)["id"])
            except Exception:
                continue
    print(f"▶ Resuming – {len(done)} profiles already enriched")

sink = open(OUT_JSONL, "a", encoding="utf-8", buffering=1)

# read the basic list *streaming* (no need for full DataFrame here)
with open(BASIC_CSV, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows   = [r for r in reader if r["id"] not in done]

print(f"▶ Need to process {len(rows)} remaining profiles")

# ─────────────────────────   main loop   ───────────────────────────
drv, prof_dir = spin_driver()

try:
    for idx, base in enumerate(rows, 1):
        extra, drv, prof_dir = grab_profile(drv, prof_dir, base["profile_url"])
        base.update(extra)

        sink.write(json.dumps(base, ensure_ascii=False) + "\n")

        if idx % 100 == 0:
            print(f"… {idx}/{len(rows)} done")
        time.sleep(random.uniform(0.25, 0.6))   # politeness
finally:
    sink.close()
    drv.quit()
    shutil.rmtree(prof_dir, ignore_errors=True)

print("✅ finished enrichment")


▶ Resuming – 1 profiles already enriched
▶ Need to process 3827 remaining profiles


KeyboardInterrupt: 

In [ ]:
# Test getting a single profile
import requests
UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64)"}
r = requests.get("https://www.datacentermap.com/usa/california/los-angeles/600-west-7th-street/", headers=UA)
print(r.status_code)

429
